# Chain-of-Thought Prompting Test

**Goal:** See what CoT looks like with Gemma 2 2B Instruct on MBPP problems.

**Reference:** [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903) (Wei et al., 2022)

**Key insight from paper:**
- Zero-shot CoT: Just add "Let's think step by step." to the prompt
- Few-shot CoT: Provide examples with step-by-step reasoning

In [7]:
# Setup
import sys
sys.path.insert(0, '..')  # Add parent directory for imports

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Check device
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

Using device: cuda


In [8]:
# Load Gemma 2 2B Instruct
model_name = "google/gemma-2-2b-it"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("Model loaded!")

Loading google/gemma-2-2b-it...


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.29it/s]


Model loaded!


In [9]:
# Load MBPP from Phase 0.1 parquet files (same as rest of codebase)
import pandas as pd
from pathlib import Path

# Load from Phase 0.1 output
phase0_1_dir = Path("../data/phase0_1")
mbpp_file = phase0_1_dir / "sae_mbpp.parquet"  # or validation_mbpp.parquet

if not mbpp_file.exists():
    # Try validation split
    mbpp_file = phase0_1_dir / "validation_mbpp.parquet"

df = pd.read_parquet(mbpp_file)
print(f"Loaded {len(df)} MBPP problems from {mbpp_file.name}")

# Convert to list of dicts for easier access (like HuggingFace dataset)
mbpp = df.to_dict('records')

# Look at first problem
problem = mbpp[0]
print("\n" + "="*60)
print("EXAMPLE PROBLEM")
print("="*60)
print(f"Task ID: {problem['task_id']}")
print(f"\nDescription:\n{problem['text']}")
print(f"\nTest cases:\n{problem['test_list']}")

Loaded 489 MBPP problems from sae_mbpp.parquet

EXAMPLE PROBLEM
Task ID: 2

Description:
Write a function to find the similar elements from the given two tuple lists.

Test cases:
['assert similar_elements((3, 4, 5, 6),(5, 7, 4, 10)) == (4, 5)'
 'assert similar_elements((1, 2, 3, 4),(5, 4, 3, 7)) == (3, 4)'
 'assert similar_elements((11, 12, 14, 13),(17, 15, 14, 13)) == (13, 14)']


## Prompt Formats

Let's compare different prompt formats:

In [10]:
def build_prompt_standard(problem):
    """Standard prompt (no CoT) - what we currently use."""
    test_cases_str = "\n".join(problem['test_list'])
    return f"""{problem['text']}

{test_cases_str}

# Solution:"""


def build_prompt_zero_shot_cot(problem):
    """Zero-shot CoT - just add 'Let's think step by step.'"""
    test_cases_str = "\n".join(problem['test_list'])
    return f"""{problem['text']}

{test_cases_str}

Let's think step by step about how to solve this problem, then write the code.

"""


def build_prompt_zero_shot_cot_v2(problem):
    """Zero-shot CoT v2 - more explicit reasoning request."""
    test_cases_str = "\n".join(problem['test_list'])
    return f"""{problem['text']}

{test_cases_str}

Before writing the code, let me think through the approach:
1. First, I'll understand what the function needs to do.
2. Then, I'll consider edge cases.
3. Finally, I'll implement the solution.

Let me reason through this:
"""


# Show the prompts
problem = mbpp[0]
print("="*60)
print("PROMPT 1: Standard (no CoT)")
print("="*60)
print(build_prompt_standard(problem))

print("\n" + "="*60)
print("PROMPT 2: Zero-shot CoT")
print("="*60)
print(build_prompt_zero_shot_cot(problem))

print("\n" + "="*60)
print("PROMPT 3: Zero-shot CoT v2 (more structured)")
print("="*60)
print(build_prompt_zero_shot_cot_v2(problem))

PROMPT 1: Standard (no CoT)
Write a function to find the similar elements from the given two tuple lists.

assert similar_elements((3, 4, 5, 6),(5, 7, 4, 10)) == (4, 5)
assert similar_elements((1, 2, 3, 4),(5, 4, 3, 7)) == (3, 4)
assert similar_elements((11, 12, 14, 13),(17, 15, 14, 13)) == (13, 14)

# Solution:

PROMPT 2: Zero-shot CoT
Write a function to find the similar elements from the given two tuple lists.

assert similar_elements((3, 4, 5, 6),(5, 7, 4, 10)) == (4, 5)
assert similar_elements((1, 2, 3, 4),(5, 4, 3, 7)) == (3, 4)
assert similar_elements((11, 12, 14, 13),(17, 15, 14, 13)) == (13, 14)

Let's think step by step about how to solve this problem, then write the code.



PROMPT 3: Zero-shot CoT v2 (more structured)
Write a function to find the similar elements from the given two tuple lists.

assert similar_elements((3, 4, 5, 6),(5, 7, 4, 10)) == (4, 5)
assert similar_elements((1, 2, 3, 4),(5, 4, 3, 7)) == (3, 4)
assert similar_elements((11, 12, 14, 13),(17, 15, 14, 13))

In [11]:
def generate(prompt, max_new_tokens=500):
    """Generate text from prompt."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # Decode only the generated part
    generated = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    return generated

## Test 1: Compare outputs on a single problem

In [12]:
# Test on first problem
problem = mbpp[0]

print("="*60)
print(f"PROBLEM: {problem['text'][:100]}...")
print("="*60)

# Standard prompt
print("\n" + "-"*40)
print("OUTPUT 1: Standard (no CoT)")
print("-"*40)
prompt1 = build_prompt_standard(problem)
output1 = generate(prompt1)
print(output1[:1000])  # First 1000 chars

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


PROBLEM: Write a function to find the similar elements from the given two tuple lists....

----------------------------------------
OUTPUT 1: Standard (no CoT)
----------------------------------------

def similar_elements(tuple1, tuple2):
    """
    Finds the similar elements from the given two tuple lists.

    Args:
        tuple1: The first tuple list.
        tuple2: The second tuple list.

    Returns:
        A list of similar elements.
    """
    similar_elements = []
    for element1 in tuple1:
        for element2 in tuple2:
            if element1 == element2:
                similar_elements.append(element1)
    return similar_elements

# Test the function
print(similar_elements((3, 4, 5, 6),(5, 7, 4, 10)))
print(similar_elements((1, 2, 3, 4),(5, 4, 3, 7)))
print(similar_elements((11, 12, 14, 13),(17, 15, 14, 13)))
```

**Explanation:**

1. **Function Definition:**
   - The code defines a function called `similar_elements` that takes two tuple lists (`tuple1` and `tuple2`

In [13]:
# Zero-shot CoT
print("-"*40)
print("OUTPUT 2: Zero-shot CoT")
print("-"*40)
prompt2 = build_prompt_zero_shot_cot(problem)
output2 = generate(prompt2, max_new_tokens=800)  # More tokens for reasoning
print(output2)  # First 1500 chars

----------------------------------------
OUTPUT 2: Zero-shot CoT
----------------------------------------
**Step 1: Understanding the Problem**

We need to find the elements that are present in both of the input tuple lists. 

**Step 2: Approach**

We can use a nested loop to compare each element in the first tuple list with each element in the second tuple list. If an element is found in both lists, we can add it to a new list.

**Step 3: Code**

```python
def similar_elements(tuple1, tuple2):
  """
  Finds the similar elements from the given two tuple lists.

  Args:
    tuple1: The first tuple list.
    tuple2: The second tuple list.

  Returns:
    A list of similar elements.
  """
  similar_elements = []
  for element1 in tuple1:
    for element2 in tuple2:
      if element1 == element2:
        similar_elements.append(element1)
  return similar_elements

# Example usage
print(similar_elements((3, 4, 5, 6),(5, 7, 4, 10)))
print(similar_elements((1, 2, 3, 4),(5, 4, 3, 7)))
print(si

In [14]:
# Zero-shot CoT v2
print("-"*40)
print("OUTPUT 3: Zero-shot CoT v2")
print("-"*40)
prompt3 = build_prompt_zero_shot_cot_v2(problem)
output3 = generate(prompt3, max_new_tokens=800)
print(output3)

----------------------------------------
OUTPUT 3: Zero-shot CoT v2
----------------------------------------
1. The function needs to find the elements that are present in both tuple lists.
2. It should handle cases where the lists have different lengths.
3. It should handle cases where the elements are not in the same order.

Now, let's write the code:

```python
def similar_elements(tuple1, tuple2):
    """
    Finds the similar elements from the given two tuple lists.

    Args:
        tuple1: The first tuple list.
        tuple2: The second tuple list.

    Returns:
        A tuple containing the similar elements.
    """
    similar_elements = []
    for element1 in tuple1:
        if element1 in tuple2:
            similar_elements.append(element1)
    return similar_elements

# Example usage
print(similar_elements((3, 4, 5, 6),(5, 7, 4, 10)))
print(similar_elements((1, 2, 3, 4),(5, 4, 3, 7)))
print(similar_elements((11, 12, 14, 13),(17, 15, 14, 13)))
```
```python
def similar_e

## Test 2: Try on a few more problems

In [15]:
# Test on problems 0, 5, 10
test_indices = [0, 5, 10]

for idx in test_indices:
    problem = mbpp[idx]
    print("\n" + "="*70)
    print(f"PROBLEM {idx}: {problem['task_id']}")
    print(f"Description: {problem['text'][:100]}...")
    print("="*70)
    
    # Generate with CoT
    prompt = build_prompt_zero_shot_cot(problem)
    output = generate(prompt, max_new_tokens=800)
    
    print("\nOUTPUT (with CoT):")
    print("-"*40)
    print(output[:1200])
    print("..." if len(output) > 1200 else "")


PROBLEM 0: 2
Description: Write a function to find the similar elements from the given two tuple lists....

OUTPUT (with CoT):
----------------------------------------
**Step 1: Understanding the Problem**

We need to find the elements that are present in both of the input tuple lists. 

**Step 2: Approach**

We can use a nested loop to compare each element in the first tuple list with each element in the second tuple list. If an element is found in both lists, we can add it to a new list.

**Step 3: Code**

```python
def similar_elements(tuple1, tuple2):
  """
  Finds the similar elements from the given two tuple lists.

  Args:
    tuple1: The first tuple list.
    tuple2: The second tuple list.

  Returns:
    A list of similar elements.
  """
  similar_elements = []
  for element1 in tuple1:
    for element2 in tuple2:
      if element1 == element2:
        similar_elements.append(element1)
  return similar_elements

# Example usage
print(similar_elements((3, 4, 5, 6),(5, 7, 4, 10

## Observations

After running the above cells, answer these questions:

1. **Does Gemma produce meaningful reasoning?** (or just goes straight to code)
2. **How long is the reasoning trace?** (tokens before code starts)
3. **Does the reasoning contain confidence expressions?** ("I think", "clearly", "maybe")
4. **Is this sufficient for faithfulness analysis?**

### Notes:
- [ ] _Write your observations here after running_

## Optional: Test with chat template

Gemma-2-2b-it may work better with its native chat template.

In [ ]:
def build_prompt_chat_cot(problem):
    """Use Gemma's chat template with CoT."""
    test_cases_str = "\n".join(problem['test_list'])
    
    user_message = f"""I need to write a Python function for this problem:

{problem['text']}

Test cases:
{test_cases_str}

Please think through this step by step:
1. What does the function need to do?
2. What are the edge cases?
3. What's the best approach?

Then provide the code."""
    
    # Use Gemma's chat template
    chat = [
        {"role": "user", "content": user_message}
    ]
    
    return tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

# Test with chat template
problem = mbpp[0]
print("PROMPT with chat template:")
print("="*60)
chat_prompt = build_prompt_chat_cot(problem)
print(chat_prompt)

In [ ]:
# Generate with chat template
print("OUTPUT with chat template:")
print("="*60)
output_chat = generate(chat_prompt, max_new_tokens=1000)
print(output_chat[:2000])

## Decision: Which prompt format to use?

Based on the outputs above, decide:

- [ ] Standard (no CoT) - if reasoning isn't meaningful
- [ ] Zero-shot CoT simple - if basic "step by step" works
- [ ] Zero-shot CoT v2 - if structured reasoning is better
- [ ] Chat template CoT - if chat format produces better reasoning